# Text and Category Cleaning

*Normalize known text variations while retaining evidence of uncertain corrections.*


## Spaces and Letter Case

Whitespace and case differences can split one category into several groups.
The string methods `.str.strip()`, `.str.title()`, and `.str.upper()` apply
the same operation to each value. Work on a copy to preserve the input.
Title case is a convention for these prepared names, not a universal rule for
personal names.


In [1]:
import pandas as pd

raw_df = pd.DataFrame(
    {
        "name": ["  jiyoung ", " HOJUN", "hana "],
        "subject": [" mathematics", "ENGLISH ", " science"],
        "city": ["seoul ", "  Busan", "  daegu "],
    }
)
clean_df = raw_df.copy()
clean_df["name"] = clean_df["name"].str.strip().str.title()
clean_df["subject"] = clean_df["subject"].str.strip().str.title()
clean_df["city"] = clean_df["city"].str.strip().str.upper()
print("Normalized text:")
print(clean_df.to_string(index=False))
print("Original first name:", repr(raw_df.loc[0, "name"]))


Normalized text:
   name     subject  city
Jiyoung Mathematics SEOUL
  Hojun     English BUSAN
   Hana     Science DAEGU
Original first name: '  jiyoung '


The names become Jiyoung, Hojun, and Hana. City labels use a single case,
and the original Jiyoung value still contains its spaces.


## Standardize Categories with a Controlled Mapping

Formatting alone cannot establish that `math` and `Mathematics` mean the same
subject. A dictionary records accepted equivalents. `.map()` returns a missing
value for an unknown key, making an unrecognized category visible for review.


In [2]:
subject_labels = pd.Series(["math", "Mathematics ", " ENG", "english", "biology"])
subject_mapping = {
    "math": "Mathematics",
    "mathematics": "Mathematics",
    "mat": "Mathematics",
    "eng": "English",
    "english": "English",
}
subject_keys = subject_labels.str.strip().str.lower()
mapped_subjects = subject_keys.map(subject_mapping)
print(
    pd.DataFrame({"raw_subject": subject_labels, "subject": mapped_subjects}).to_string(
        index=False
    )
)
print("Unmapped labels:", subject_labels[mapped_subjects.isna()].tolist())


 raw_subject     subject
        math Mathematics
Mathematics  Mathematics
         ENG     English
     english     English
     biology         NaN
Unmapped labels: ['biology']


Four labels map to the two accepted subjects. Biology remains unresolved
because this lookup does not define it; it has not been silently relabeled.


## Character Filtering Has Limits

The source examples remove punctuation and digits from prepared ASCII names.
Use that rule only where those characters are known entry noise. Real names
can contain accents, hyphens, apostrophes, or non-Latin characters. Filtering
also cannot reconstruct letters that are already missing.


In [3]:
raw_names = pd.Series(["J!iyoung", "Ho_jun*", "Ha^na", "J!yo2ung"])


def keep_ascii_letters_and_spaces(text):
    return "".join(
        character
        for character in text
        if character.isascii() and (character.isalpha() or character == " ")
    )


candidate_names = raw_names.map(keep_ascii_letters_and_spaces).str.strip().str.title()
print(
    pd.DataFrame({"raw_name": raw_names, "candidate_name": candidate_names}).to_string(
        index=False
    )
)


raw_name candidate_name
J!iyoung        Jiyoung
 Ho_jun*          Hojun
   Ha^na           Hana
J!yo2ung         Jyoung


The known noisy spelling `J!iyoung` becomes Jiyoung. However, `J!yo2ung`
becomes Jyoung, not Jiyoung: deleting noise cannot recover the missing `i`.
Keep this candidate separate until the source identity is confirmed.


Normalize formatting consistently, use explicit category mappings, and retain raw text when a correction could change identity or meaning.
